<a href="https://colab.research.google.com/github/Sirisha05n4/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q google-genai pydantic

import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [8]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [19]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    # Manual guard rail: If the input is completely empty, trigger the validation error manually
    if not raw_text.strip():
        raise ValidationError.from_exception_data(
            title="Resume",
            line_errors=[{"type": "missing", "loc": ("name",), "input": raw_text}]
        )

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [11]:
import os

for root, dirs, files in os.walk('.'):
    for file in files:
        print(os.path.join(root, file))

./.config/.last_update_check.json
./.config/.last_survey_prompt.yaml
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/config_sentinel
./.config/active_config
./.config/.last_opt_in_prompt.yaml
./.config/default_configs.db
./.config/gce
./.config/configurations/config_default
./.config/logs/2026.06.04/13.32.39.344962.log
./.config/logs/2026.06.04/13.32.21.210570.log
./.config/logs/2026.06.04/13.32.18.735754.log
./.config/logs/2026.06.04/13.32.02.654775.log
./.config/logs/2026.06.04/13.32.38.346437.log
./.config/logs/2026.06.04/13.31.42.499627.log
./sample_data/anscombe.json
./sample_data/README.md
./sample_data/california_housing_train.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv


In [13]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [16]:
resume_data = """Ravi Kumar
Email: ravi@gmail.com
Phone: 9876543210

Education:
B.Tech, JNTU, 2024

Skills:
Python, SQL, Git, HTML, CSS, JavaScript

Projects:
Resume Parser

Experience:
1 year

---

Sneha Reddy
Email: sneha@gmail.com

Education:
B.Sc, Andhra University, 2023

Skills:
Python, Excel, Power BI, SQL, Communication, Leadership

Projects:
Sales Dashboard

Experience:
0.5 years

---

Arun Pillai
Email: arun@gmail.com
Phone: 9123456789

Education:
B.Tech, Kerala University, 2024

Skills:
Python, Java, C++, SQL, Git, Linux, Docker, Flask, HTML

Projects:
Chatbot System

Experience:
1 year"""

# This writes the text above into the file
with open("sample_resumes.txt", "w") as f:
    f.write(resume_data)

print("Success! File 'sample_resumes.txt' has been created.")

Success! File 'sample_resumes.txt' has been created.


In [17]:
with open('sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 3 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 1.0 years exp

Résumé 2: FAILED — ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Résumé 3: FAILED — ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi@gmail.com",
  "phone": "9876543210",
  "education": [
    {
      "degree": "B.Tech",
      "institution": "JNTU",
      "year": 2024
    }
  ],
  "skills": [
    "Python",
    "SQL",
    "Git",
    "HTML",
    "CSS",
    "JavaScript"
  ],
  "projects": [
    "Resume Parser"
  ],
  "experience_years": 1.0
}


In [20]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: ValidationError
Message: 1 validation error for Resume
name
  Field required [type=missing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
